# 02 — Обучение одной CNN на RuASD (GPU)
Блокнот использует 38 017 подготовленных FLAC из notebook 01. Log-Mel вычисляются на лету. `test_unseen` не участвует в обучении, TensorBoard, ранней остановке или выборе checkpoint.

In [ ]:
# 1. Среда, репозиторий и зависимости
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Unvand/audio-deepfake-detector.git'
REPO_DIR = Path('/content/audio-deepfake-detector')
if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git', 'clone', '-q', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements-colab-train.txt')], check=True)
    os.chdir(REPO_DIR)
print('Python:', sys.version)

In [ ]:
# 2. Drive, GPU и неизменяемый manifest подготовленных сегментов
import pandas as pd
import torch
from IPython.display import clear_output, display

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
PREPARED_ROOT = Path('/content/drive/MyDrive/VKR_RuASD/prepared_v1')
TRAINING_ROOT = Path('/content/drive/MyDrive/VKR_RuASD/training_v1')
RUNS_ROOT = TRAINING_ROOT / 'runs'
SEGMENTS_MANIFEST = PREPARED_ROOT / 'ruasd_segments_manifest.csv'
if not SEGMENTS_MANIFEST.exists():
    raise FileNotFoundError('Сначала выполните notebook 01: ' + str(SEGMENTS_MANIFEST))
if not torch.cuda.is_available():
    raise RuntimeError('Выберите Runtime → Change runtime type → GPU')
print('GPU:', torch.cuda.get_device_name(0))
segments = pd.read_csv(SEGMENTS_MANIFEST)
if segments['segment_id'].duplicated().any():
    raise ValueError('Найдены повторяющиеся segment_id')
if set(segments['split'].unique()) != {'train', 'validation', 'test_unseen'}:
    raise ValueError('Неожиданный набор split')
display(pd.crosstab(segments['split'], segments['label'], margins=True))
print('Сегментов:', len(segments))

## Настройки обучения CNN
Виджеты ниже непосредственно создают `MelConfig` и `TrainingConfig`. Кнопка проверки также предупреждает о несовместимом checkpoint. Изменение виджетов само по себе обучение не запускает.

In [ ]:
# 3. Интерактивная панель конфигурации
import json
from dataclasses import asdict

import ipywidgets as widgets

if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

from audio_deepfake_detector.src.cnn_training import (
    RuASDSegmentDataset,
    assert_checkpoint_compatible,
    compute_pos_weight,
    configs_from_mapping,
    create_spectrogram_figures,
    evaluate_checkpoint,
    make_loaders,
    resolve_run_directory,
    train_cnn,
)

controls = {
    'batch_size': widgets.Dropdown(options=[8, 16, 32, 64], value=32, description='Batch size'),
    'epochs': widgets.IntSlider(value=20, min=1, max=100, description='Epochs'),
    'learning_rate': widgets.Dropdown(options=[1e-4, 3e-4, 1e-3, 3e-3], value=1e-3, description='LR'),
    'weight_decay': widgets.FloatText(value=1e-4, description='Weight decay'),
    'early_stopping_patience': widgets.IntSlider(value=5, min=1, max=20, description='Patience'),
    'num_workers': widgets.Dropdown(options=[0, 1, 2, 4], value=2, description='Workers'),
    'seed': widgets.IntText(value=42, description='Seed'),
    'balance_classes': widgets.Checkbox(value=True, description='Балансировка pos_weight'),
    'use_amp': widgets.Checkbox(value=torch.cuda.is_available(), description='Mixed precision AMP'),
    'n_fft': widgets.Dropdown(options=[512, 1024, 2048], value=1024, description='FFT size'),
    'win_length': widgets.Dropdown(options=[512, 1024, 2048], value=1024, description='Window'),
    'hop_length': widgets.IntText(value=320, description='Hop length'),
    'n_mels': widgets.IntSlider(value=80, min=32, max=160, description='Mel bins'),
    'f_min': widgets.FloatText(value=20.0, description='Min Hz'),
    'f_max': widgets.FloatText(value=16000.0, description='Max Hz'),
}
run_id_widget = widgets.Text(value='baseline_v1', description='run_id')
resume_widget = widgets.Checkbox(value=True, description='Resume training')
show_button = widgets.Button(description='Показать текущую конфигурацию', button_style='info')
config_output = widgets.Output()

def current_configs():
    values = {name: widget.value for name, widget in controls.items()}
    return configs_from_mapping(values)

def show_configuration(_=None):
    with config_output:
        clear_output()
        try:
            mel_config, training_config = current_configs()
            run_dir = resolve_run_directory(TRAINING_ROOT, run_id_widget.value)
            weight = compute_pos_weight(segments)
            payload = {
                'run_directory': str(run_dir),
                'resume': resume_widget.value,
                'computed_train_pos_weight_REAL_div_FAKE': weight,
                'applied_pos_weight': weight if training_config.balance_classes else 1.0,
                'mel_config': asdict(mel_config),
                'training_config': asdict(training_config),
            }
            print(json.dumps(payload, ensure_ascii=False, indent=2))
            checkpoint_path = run_dir / 'last_checkpoint.pth'
            if resume_widget.value and checkpoint_path.exists():
                checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
                assert_checkpoint_compatible(checkpoint, mel_config, training_config)
                print('Checkpoint совместим; продолжение с эпохи', checkpoint['epoch'] + 2)
        except Exception as error:  # noqa: BLE001 — ошибка показывается в панели
            print('ОШИБКА КОНФИГУРАЦИИ:', error)

show_button.on_click(show_configuration)
training_box = widgets.VBox([
    widgets.HTML('<b>Обучение</b>'),
    *[controls[key] for key in ['batch_size', 'epochs', 'learning_rate', 'weight_decay', 'early_stopping_patience', 'num_workers', 'seed', 'balance_classes', 'use_amp']],
    widgets.HTML('<b>Mel-спектрограмма (sample rate фиксирован: 32 000 Гц)</b>'),
    *[controls[key] for key in ['n_fft', 'win_length', 'hop_length', 'n_mels', 'f_min', 'f_max']],
    widgets.HTML('<b>Эксперимент</b>'), run_id_widget, resume_widget, show_button, config_output,
])
display(training_box)

In [ ]:
# 4. Предварительная проверка одного batch: train/validation, форма и NaN/Inf
preflight_button = widgets.Button(description='Проверить один batch', button_style='warning')
preflight_output = widgets.Output()

def run_preflight(_):
    with preflight_output:
        clear_output()
        try:
            mel_config, training_config = current_configs()
            train_loader, validation_loader = make_loaders(
                segments, PREPARED_ROOT, mel_config, training_config
            )
            for name, loader in [('train', train_loader), ('validation', validation_loader)]:
                log_mel, labels, sample_ids = next(iter(loader))
                print(name, 'split=', loader.dataset.split, 'shape=', tuple(log_mel.shape),
                      'finite=', bool(torch.isfinite(log_mel).all()),
                      'labels=', sorted(labels.unique().tolist()),
                      'sample_ids=', len(sample_ids))
            print('test_unseen не создавался и не читался.')
        except Exception as error:  # noqa: BLE001 — ошибка показывается в панели
            print('ОШИБКА PREFLIGHT:', error)

preflight_button.on_click(run_preflight)
display(preflight_button, preflight_output)

In [ ]:
# 5. Реальные log-Mel REAL/FAKE до стандартизации
mel_button = widgets.Button(description='Показать реальные Mel', button_style='info')
mel_output = widgets.Output()

def show_real_mels(_):
    with mel_output:
        clear_output()
        try:
            mel_config, training_config = current_configs()
            dataset = RuASDSegmentDataset(segments, PREPARED_ROOT, 'train', mel_config)
            figures = create_spectrogram_figures(
                dataset, training_config.spectrogram_examples_per_class
            )
            for tag, figure in figures:
                print(tag)
                display(figure)
        except Exception as error:  # noqa: BLE001 — ошибка показывается в панели
            print('ОШИБКА MEL:', error)

mel_button.on_click(show_real_mels)
display(mel_button, mel_output)

## Мониторинг обучения — TensorBoard
События сохраняются на Google Drive внутри каждого `runs/<run_id>/tensorboard`. Повторный запуск этой ячейки после отключения открывает ранее сохранённые события.

In [ ]:
# 6. TensorBoard для всех run_id
from tensorboard import notebook as tensorboard_notebook

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
tensorboard_notebook.start(f'--logdir {RUNS_ROOT} --port 6006')

## Запуск и возобновление обучения
Кнопка ниже — единственное действие, запускающее обучение. Она повторно читает текущие значения всех виджетов. При несовместимом checkpoint выполнение останавливается с рекомендацией выбрать новый `run_id`.

In [ ]:
# 7. Явная кнопка запуска — изменение настроек ничего не запускает
start_button = widgets.Button(description='Начать обучение', button_style='danger', icon='play')
training_output = widgets.Output()

def start_training(_):
    start_button.disabled = True
    with training_output:
        clear_output()
        try:
            mel_config, training_config = current_configs()
            run_dir = resolve_run_directory(TRAINING_ROOT, run_id_widget.value)
            print('Запуск:', run_dir)
            _model, history = train_cnn(
                PREPARED_ROOT, run_dir, mel_config, training_config,
                resume=resume_widget.value,
            )
            print('Завершённых эпох:', len(history))
            print('Best:', run_dir / 'best_model.pth')
            print('Last:', run_dir / 'last_checkpoint.pth')
        except Exception as error:  # noqa: BLE001 — ошибка показывается в панели
            print('ОБУЧЕНИЕ НЕ ЗАПУЩЕНО/ОСТАНОВЛЕНО:', type(error).__name__, error)
        finally:
            start_button.disabled = False

start_button.on_click(start_training)
display(start_button, training_output)

In [ ]:
# 8. Таблица сохранённой истории выбранного run_id
history_button = widgets.Button(description='Обновить историю')
history_output = widgets.Output()

def show_history(_):
    with history_output:
        clear_output()
        run_dir = resolve_run_directory(TRAINING_ROOT, run_id_widget.value)
        history_path = run_dir / 'training_history.json'
        if history_path.exists():
            payload = json.loads(history_path.read_text(encoding='utf-8'))
            history = payload.get('epochs', []) if isinstance(payload, dict) else payload
            display(pd.json_normalize(history))
        else:
            print('История ещё не создана:', history_path)

history_button.on_click(show_history)
display(history_button, history_output)

## Финальная оценка test_unseen
Выполняйте только после окончательного выбора модели и параметров по validation. Эта кнопка не связана с обучением и никогда не нажимается автоматически.

In [ ]:
# 9. Отдельное явное действие для test_unseen
test_button = widgets.Button(description='ФИНАЛЬНО оценить test_unseen', button_style='danger')
test_output = widgets.Output()

def evaluate_test_unseen(_):
    with test_output:
        clear_output()
        try:
            _, training_config = current_configs()
            run_dir = resolve_run_directory(TRAINING_ROOT, run_id_widget.value)
            best_model = run_dir / 'best_model.pth'
            if not best_model.exists():
                raise FileNotFoundError(best_model)
            metrics = evaluate_checkpoint(
                PREPARED_ROOT, best_model, split='test_unseen',
                batch_size=training_config.batch_size,
                num_workers=training_config.num_workers,
            )
            print(json.dumps(metrics, ensure_ascii=False, indent=2))
            (run_dir / 'test_unseen_metrics.json').write_text(
                json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8'
            )
        except Exception as error:  # noqa: BLE001 — ошибка показывается в панели
            print('ОШИБКА ФИНАЛЬНОЙ ОЦЕНКИ:', error)

test_button.on_click(evaluate_test_unseen)
display(test_button, test_output)